In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from pathlib import Path
import os

os.makedirs('figures', exist_ok=True)

DIR = Path('data')
EVAL_COLS = [f'eval_{i:02d}' for i in range(1, 15)]

RUNS = [
    ('42226_1_blind',    '42226_1_blind_results.tsv',    'Blind 1',    '#2166ac', 'solid'),
    ('42526_1_blind',    '42526_1_blind_results.tsv',    'Blind 2',    '#6baed6', 'dashed'),
    ('42326_1_informed', '42326_1_informed_results.tsv', 'Informed 1', '#1a9641', 'solid'),
    ('42326_2_informed', '42326_2_informed_results.tsv', 'Informed 2', '#74c476', 'dashed'),
]

dfs = {}
for key, fname, label, color, ls in RUNS:
    df = pd.read_csv(DIR / fname, sep='\t')
    df['exp_num'] = np.arange(1, len(df) + 1)
    dfs[key] = df

# Parse human baseline scores from strategies.md Table 1
strat_text = (DIR / 'strategies.md').read_text()
t1 = strat_text[strat_text.index('## Table 1'):strat_text.index('## Table 2')]
table_lines = [l for l in t1.splitlines() if l.startswith('|') and not l.startswith('|---')]
header = [h.strip() for h in table_lines[0].split('|')[1:-1]]
strat_df = pd.DataFrame(
    [dict(zip(header, [v.strip() for v in l.split('|')[1:-1]])) for l in table_lines[1:]]
)
for col in EVAL_COLS:
    strat_df[col] = pd.to_numeric(strat_df[col])

human_best_per_eval = strat_df[EVAL_COLS].max()
human_min_per_eval  = strat_df[EVAL_COLS].min()
human_best_eval01   = human_best_per_eval['eval_01']
random_floor_eval01 = strat_df.loc[strat_df['strategy'] == 'synth_oracle', 'eval_01'].values[0]

print(f'Human best eval_01:   {human_best_eval01:.4f}  (dhs_topic)')
print(f'Random floor eval_01: {random_floor_eval01:.4f}  (synth_oracle)')
for key, fname, label, color, ls in RUNS:
    best = dfs[key]['eval_01'].max()
    best_exp = dfs[key].loc[dfs[key]['eval_01'].idxmax(), 'experiment']
    print(f'{label} best: {best:.4f}  ({best_exp})')

In [ ]:
# ── Fig 3: Composite — strip plot (top) + blind/informed trajectories (bottom) ──

fig = plt.figure(figsize=(11, 9.5), dpi=300)
gs  = gridspec.GridSpec(2, 2, height_ratios=[1.6, 1], hspace=0.42, wspace=0.08)
ax_strip = fig.add_subplot(gs[0, :])
ax_blind = fig.add_subplot(gs[1, 0])
ax_inf   = fig.add_subplot(gs[1, 1], sharey=ax_blind)

# Top: strip plot
rng2 = np.random.RandomState(42)
JITTER = 0.16
HUMAN_X  = 0.0
RUN_XS   = [1.6, 2.4, 3.4, 4.2]
BLIND_MID    = (RUN_XS[0] + RUN_XS[1]) / 2
INFORMED_MID = (RUN_XS[2] + RUN_XS[3]) / 2
ax = ax_strip

human_vals = strat_df['eval_01'].values
jx = rng2.uniform(-JITTER, JITTER, len(human_vals))
ax.scatter(HUMAN_X + jx, human_vals, color='#999999', s=28, alpha=0.85, zorder=3, linewidths=0)
for (key, fname, label, color, ls), rx in zip(RUNS, RUN_XS):
    df = dfs[key]
    best_idx = df['eval_01'].idxmax()
    jx = rng2.uniform(-JITTER, JITTER, len(df))
    mask = df.index != best_idx
    ax.scatter(rx + jx[mask], df.loc[mask, 'eval_01'],
               color=color, s=22, alpha=0.55, zorder=3, linewidths=0)
    ax.scatter(rx, df.loc[best_idx, 'eval_01'],
               color=color, s=200, marker='*', zorder=5, edgecolors='white', linewidths=0.7)

ax.axhline(human_best_eval01,   color='#444444', linestyle='--', linewidth=1.1, zorder=2)
ax.axhline(random_floor_eval01, color='#888888', linestyle=':',  linewidth=1.0, zorder=2)
ax.axvline(0.9,  color='#e0e0e0', linewidth=1.0, zorder=1)
ax.axvline(2.95, color='#e0e0e0', linewidth=1.0, zorder=1)
ax.text(HUMAN_X,      0.805, 'Human\nbaselines', ha='center', va='bottom', fontsize=8,   color='#666666', style='italic')
ax.text(BLIND_MID,    0.805, 'Blind',             ha='center', va='bottom', fontsize=8.5, color='#2166ac', fontweight='bold')
ax.text(INFORMED_MID, 0.805, 'Informed',          ha='center', va='bottom', fontsize=8.5, color='#1a9641', fontweight='bold')
ax.text(4.55, human_best_eval01  + 0.003, f'Human best ({human_best_eval01:.3f})',
        va='bottom', ha='right', fontsize=7.5, color='#444444')
ax.text(4.55, random_floor_eval01 - 0.003, f'Random floor ({random_floor_eval01:.3f})',
        va='top',    ha='right', fontsize=7.5, color='#999999')
ax.set_xticks([HUMAN_X] + RUN_XS)
ax.set_xticklabels(['', 'Blind 1', 'Blind 2', 'Informed 1', 'Informed 2'], fontsize=8.5)
for tick, c in zip(ax.get_xticklabels(), ['black', '#2166ac', '#6baed6', '#1a9641', '#74c476']):
    tick.set_color(c)
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#999999', markersize=6,
           label='Human-designed strategy (50k, 5 seeds)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#888888', markersize=6,
           alpha=0.6, label='Agent experiment'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='#888888', markersize=10,
           label='Best per run'),
]
ax.legend(handles=legend_elements, fontsize=7.5, loc='lower right', framealpha=0.9, borderpad=0.7)
ax.set_xlim(-0.55, 4.75); ax.set_ylim(0.5, 0.80)
ax.set_ylabel('Pearson r vs ground truth (eval_01)', fontsize=10)
ax.yaxis.grid(True, color='#eeeeee', zorder=0); ax.set_axisbelow(True)
ax.spines[['top', 'right', 'bottom']].set_visible(False)
ax.tick_params(bottom=False)

# Bottom: blind / informed split panels
groups = [
    (ax_blind, 'Blind',    '#2166ac', [r for r in RUNS if 'blind'    in r[0]]),
    (ax_inf,   'Informed', '#1a9641', [r for r in RUNS if 'informed' in r[0]]),
]
for ax, group_label, title_color, group_runs in groups:
    for key, fname, label, color, ls in group_runs:
        df = dfs[key]
        best_idx = df['eval_01'].idxmax()
        ax.plot(df['exp_num'], df['eval_01'], color=color, linestyle=ls,
                linewidth=1.4, zorder=3, label=label)
        ax.scatter(df['exp_num'], df['eval_01'],
                   color=color, s=22, alpha=0.7, zorder=4, linewidths=0)
        ax.scatter(df.loc[best_idx, 'exp_num'], df.loc[best_idx, 'eval_01'],
                   color=color, s=280, marker='*', zorder=6, edgecolors='white', linewidths=0.8)
    ax.axhline(human_best_eval01,   color='#444444', linestyle='--', linewidth=1.1, zorder=2,
               label=f'Human best ({human_best_eval01:.3f})')
    ax.axhline(random_floor_eval01, color='#888888', linestyle=':',  linewidth=1.3, zorder=2,
               label=f'Random floor ({random_floor_eval01:.3f})')
    ax.set_xlim(0.5, 30.5); ax.set_ylim(0.5, 0.80)
    ax.set_title(group_label, fontsize=10, fontweight='bold', color=title_color)
    ax.set_xlabel('Experiment number', fontsize=9)
    ax.legend(fontsize=7.5, loc='lower right', framealpha=0.9, borderpad=0.6)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0); ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
ax_blind.set_ylabel('Pearson r vs ground truth (eval_01)', fontsize=9)
plt.setp(ax_inf.get_yticklabels(), visible=False)

fig.savefig('figures/fig3_agent_composite.pdf', bbox_inches='tight')
fig.savefig('figures/fig3_agent_composite.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig3_agent_composite.{pdf,png}')

In [ ]:
# ── Fig 4: Per-eval generalisation — best agent design per run across 14 eval sets ──

xs = np.arange(14)

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=300)

ax.fill_between(xs, human_min_per_eval[EVAL_COLS].values, human_best_per_eval[EVAL_COLS].values,
                color='#cccccc', alpha=0.45, zorder=1, label='Human baseline range')
ax.plot(xs, human_best_per_eval[EVAL_COLS].values,
        color='#666666', linestyle='--', linewidth=1.3, zorder=2, label='Human best per eval')

for key, fname, label, color, ls in RUNS:
    best_row = dfs[key].loc[dfs[key]['eval_01'].idxmax(), EVAL_COLS].astype(float).values
    ax.plot(xs, best_row, color=color, linestyle=ls, linewidth=1.8,
            marker='o', markersize=4.5, zorder=3, label=label)

ax.set_xticks(xs)
ax.set_xticklabels(EVAL_COLS, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Pearson r vs ground truth', fontsize=10)
ax.set_title('Best agent design per run — generalisation across 14 eval sets', fontsize=10.5)
ax.legend(fontsize=8, loc='lower center', ncol=3, framealpha=0.9, borderpad=0.7)
ax.yaxis.grid(True, color='#eeeeee', zorder=0)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()

fig.savefig('figures/fig4_generalisation.pdf')
fig.savefig('figures/fig4_generalisation.png', dpi=300)
plt.show()
print('Saved: figures/fig4_generalisation.{pdf,png}')